In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# JSONL

In [0]:
schema = StructType([
    StructField('id',IntegerType(),nullable=False),
    StructField('name',StringType(),True),
    StructField('salary',IntegerType(),False)
])

jsonL = spark.read.format("json").schema(schema).load("/Volumes/practice1234flm/default/json/JSONL.json")

jsonL.printSchema()
display(jsonL)

# JSON multiline

In [0]:
json_multiline = spark.read.format('json').option('multiline',True).load('/Volumes/practice1234flm/default/json/JSON_multiline.json')
print(json_multiline)
display(json_multiline)

# JSON nested

In [0]:

json_nested = spark.read.format('json').load('/Volumes/practice1234flm/default/json/JSON_nested.json')
final = json_nested.withColumn('city',F.col("address.city")) \
    .withColumn('state',F.col("address.state")) \
        .drop("address")

# final = json_nested.select('*', F.explode("address").alias('addresses'))
display(final)

# JSON array

In [0]:
from pyspark.sql import functions as F
json_array = spark.read.format('json').load('/Volumes/practice1234flm/default/json/JSON_arrays.json')
# json_array = json_array.withColumn('PrimarySkill', F.col('skills')[0]) \
#     .withColumn('SecondSkill', F.col('skills')[1]) \
#         .drop("skills")
# display(json_array)

json_array = json_array.select('*',F.explode(F.col('skills')).alias('skill')) \
    .drop('skills')

display(json_array)

# JSON Complex

In [0]:
json_complex = spark.read.format('json').option('multiline',True).load('/Volumes/practice1234flm/default/json/JSON_complex.json')
final_temp = json_complex.withColumn('customer_id',F.col('customer.customer_id')) \
    .withColumn('email', F.col('customer.email')) \
        .withColumn('name',F.col('customer.name')) \
            .withColumn('city',F.col("shipping_address.city")) \
                .withColumn('country',F.col("shipping_address.country")) \
                    .withColumn('state',F.col("shipping_address.state")) \
                        .drop("customer","shipping_address")
display(final_temp)

In [0]:
items = final.select(F.col("items")).collect()
json_list = [row.asDict() for row in items]
schema = StructType([
    StructField('price',DoubleType()),
    StructField('product_id',IntegerType()),
    StructField('product_name',StringType()),
    StructField('quantity',IntegerType())
])
# print(json_list)
products = spark.createDataFrame(json_list[0]['items'], schema)
display(products)

In [0]:
final = final_temp.select('*', F.explode("items").alias('item')).select('*',F.col('item.product_id'),F.col('item.product_name'),F.col('item.price'),F.col('item.quantity')) \
    .drop('items','item')

display(final)